In [13]:
!pip install xgboost

In [15]:
!pip install imbalanced-learn

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import streamlit as st
import joblib
import seaborn as sns


from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    average_precision_score,
    roc_auc_score
)

from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import ADASYN
from sklearn.inspection import permutation_importance
from imblearn.under_sampling import RandomUnderSampler

In [19]:
data = pd.read_csv("data.csv")

target_col = "Bankrupt?"

X = data.drop(columns=[target_col])
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

base_scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

sample_weights = np.where(y_train == 1, base_scale_pos_weight, 1.0)

In [26]:
pipe = ImbPipeline(steps=[
    (
        "under",
        RandomUnderSampler(random_state=42)
    ),
    (
        "smote",
        SMOTE(random_state=42)
    ),
    (
        "model",
        XGBClassifier(
            eval_metric='aucpr',
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            n_estimators=400,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist"
        )
    )
])

param_grid = {
    'under__sampling_strategy': [0.05, 0.08],

    'smote__sampling_strategy': [0.25, 0.30, 0.35],
    'smote__k_neighbors': [3, 5],

    'model__max_depth': [4, 5, 6],
    'model__learning_rate': [0.03, 0.05],
    'model__scale_pos_weight': [1.5, 2.0]
}


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scorer = make_scorer(f1_score, pos_label=1)

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring=scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=False,
    refit=True
)

grid_search.fit(X_train, y_train)

print("\nBest params:")
print(grid_search.best_params_)
print(f"F1-score (CV): {grid_search.best_score_:.4f}")

best_model1 = grid_search.best_estimator_

probs = best_model1.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\nThreshold: {best_threshold:.4f}")

y_pred_opt = (probs >= best_threshold).astype(int)

print("\nClassification Report (XGBoost + UnderSampling + SMOTE + GridSearch + Optimal Threshold)")
print(classification_report(y_test, y_pred_opt, digits=4))

Fitting 5 folds for each of 144 candidates, totalling 720 fits

Best params:
{'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__scale_pos_weight': 1.5, 'smote__k_neighbors': 3, 'smote__sampling_strategy': 0.25, 'under__sampling_strategy': 0.08}
F1-score (CV): 0.4927

Threshold: 0.7305

Classification Report (XGBoost + UnderSampling + SMOTE + GridSearch + Optimal Threshold)
              precision    recall  f1-score   support

           0     0.9870    0.9803    0.9837      1320
           1     0.5094    0.6136    0.5567        44

    accuracy                         0.9685      1364
   macro avg     0.7482    0.7970    0.7702      1364
weighted avg     0.9716    0.9685    0.9699      1364



In [29]:
# Uloženie najlepšieho modelu do súboru
joblib.dump(best_model1, "bankruptcy_model.pkl")

# Uloženie optimálneho prahu klasifikácie
joblib.dump(best_threshold, "best_threshold.pkl")

# Uloženie názvov použitých premenných
joblib.dump(X_train.columns.tolist(), "feature_names.pkl")

# Uloženie testovacích vstupných dát do CSV súboru
X_test.to_csv("X_test.csv", index=False)

# Uloženie testovacích cieľových hodnôt do CSV súboru
y_test.to_csv("y_test.csv", index=False)